In [48]:
import pandas as pd
scores = pd.read_csv("results_scored.csv")


## Missing result Imputation and Score Rellocation
Algorithm
----
    1. groupby 'Query' and 'Rank"
    2. check if all four methods are recorded
        - 'BERT', 'SBERT', 'TF-IDF + GloVe', 'TF-IDF'
        - record the number of missing methods
    3. if there is at least one method not being represented
        1. add rows for the missing methods with giving the score as 1
        2. increase the other method's score with the number of previously missing methods
        3. ensure thatno method has a score higher than 3
    5. save to a new Dataframe 

In [49]:
import pandas as pd

EXPECTED_METHODS = {
    "BERT",
    "SBERT",
    "TF-IDF + GloVe",
    "TF-IDF"
}

new_groups = []
augmented_scores = pd.DataFrame()

for (query, rank), group in scores.groupby(["Query", "Rank"]):

    group = group.copy()

    # Mark original rows
    group["Imputed"] = False

    present_methods = set(group["Method"])
    missing_methods = EXPECTED_METHODS - present_methods
    n_missing = len(missing_methods)

    if n_missing > 0:

        # Increase existing scores
        group["Score"] = (group["Score"] + n_missing).clip(upper=3)

        for method in missing_methods:

            new_row = {col: pd.NA for col in scores.columns}

            # Required fields
            new_row["Query"] = query
            new_row["Rank"] = rank
            new_row["Method"] = method
            new_row["Score"] = 0

            # Explicitly null out retrieval data
            new_row["Similarity Score (%)"] = pd.NA
            new_row["Text"] = pd.NA
            new_row["Psalm Num"] = pd.NA
            new_row["Verse"] = pd.NA
            new_row["Scored"] = pd.NA
            new_row["Letter"] = pd.NA

            new_row["Imputed"] = True

            group = pd.concat(
                [group, pd.DataFrame([new_row])],
                ignore_index=True
            )

    new_groups.append(group)

augmented_scores = pd.concat(new_groups, ignore_index=True)

# Verification
print(
    augmented_scores
    .groupby(["Query", "Rank"])["Method"]
    .nunique()
    .value_counts()
)

print(
    augmented_scores["Imputed"]
    .value_counts()
)

Method
4    420
Name: count, dtype: int64
Imputed
False    10783
True        70
Name: count, dtype: int64


In [50]:
augmented_scores[augmented_scores["Query"] == 'struggle']

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,Imputed
10079,struggle,BERT,1,35.08,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,A,1,False
10080,struggle,SBERT,1,96.24,Psalter,150,Praise God in His holy ones; praise Him in the...,False,B,2,False
10081,struggle,TF-IDF,1,0.0,Psalter,150,Praise God in His holy ones; praise Him in the...,False,C,3,False
10082,struggle,BERT,1,35.08,Psalter,42,"Judge me, God, and give judgment in my cause a...",True,A,3,False
10083,struggle,SBERT,1,96.24,Psalter,150,Praise God in His holy ones; praise Him in the...,True,B,1,False
...,...,...,...,...,...,...,...,...,...,...,...
10323,struggle,SBERT,5,95.96,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,True,B,3,False
10324,struggle,BERT,5,30.93,Psalter,1,Blessed is the man that hath not walked in the...,True,A,3,False
10325,struggle,SBERT,5,95.96,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,True,B,2,False
10326,struggle,TF-IDF,5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,True


In [51]:
query = "struggle"

orig = scores[scores["Query"] == query]
aug = augmented_scores[augmented_scores["Query"] == query]

print(f"Query: '{query}'")
print("-" * 40)
print(f"Original rows:              {len(orig)}")
print(f"Original unique rows:       {len(orig.drop_duplicates())}")
print(f"Original duplicates:        {len(orig) - len(orig.drop_duplicates())}")
print()
print(f"Augmented rows:             {len(aug)}")
print(f"Augmented unique rows:      {len(aug.drop_duplicates())}")
print(f"Augmented duplicates:       {len(aug) - len(aug.drop_duplicates())}")

Query: 'struggle'
----------------------------------------
Original rows:              240
Original unique rows:       42
Original duplicates:        198

Augmented rows:             249
Augmented unique rows:      51
Augmented duplicates:       198


In [52]:
# Saving new Data as scores
scores = augmented_scores

In [53]:
scores.shape

(10853, 11)

In [54]:
scores = scores.drop_duplicates()
scores.shape

(4911, 11)

In [55]:
duplicates = scores[
    scores.duplicated(
        subset=['Query', 'Method', 'Rank'],
        keep=False
    )
].sort_values(['Query', 'Method', 'Rank'])


In [56]:
scores = scores.drop_duplicates(
    subset=['Query', 'Method', 'Rank'],
    keep='first'
)

In [57]:
scores.shape

# This is what we needed

(1680, 11)

In [58]:
scores.head()

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score,Imputed
0,How does the psalmist describe God's compassi...,TF-IDF + GloVe,1,16.0,Bible,79,1For the End concerning things that shall be c...,False,A,0,False
1,How does the psalmist describe God's compassi...,BERT,1,75.36,Bible,18,For the End a psalm by David The heavens decla...,False,B,3,False
2,How does the psalmist describe God's compassi...,SBERT,1,97.46,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,False,C,2,False
3,How does the psalmist describe God's compassi...,TF-IDF,1,14.64,Psalter,84,"Lord, Thou hast been favourable unto Thy land;...",False,D,1,False
8,How does the psalmist describe God's compassi...,TF-IDF + GloVe,2,13.98,Bible,122,1An ode of ascents Ilift my eyes to You Whodwe...,False,A,2,False


In [59]:
scores.to_csv("clean_llm_scores.csv")

-  `human_scores` - is the results from human annotators
- `human_seen_llm_results` - is the results scored by the llm in the pairwise comparisons 

In [11]:
human_scores = pd.read_csv("human_scores.csv")
human_scores

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...,...
855,941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p09,8
856,942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p03,1
857,943,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2
858,944,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7


In [12]:
human_scores = human_scores.rename(columns={'numbered_result':'Rank'})

In [13]:
human_seen_results = pd.read_csv("results_from_humans_scored.csv")
#human_seen_results

In [14]:
human_seen_llm_results = human_seen_results.drop_duplicates(
    subset=['Query', 'Method', 'Rank'],
    keep='first'
)

human_seen_llm_results = human_seen_llm_results.sort_values(
    by='Query'
)

human_seen_llm_results

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored,Letter,Score
79,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,False,A,3
80,Create in me a clean heart,BERT,3,68.69,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",False,B,1
81,Create in me a clean heart,SBERT,3,95.62,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,False,C,2
82,Create in me a clean heart,TFIDF,3,14.20,Psalter,23,"The earth is the Lord's, and the fulness there...",False,D,0
102,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,False,A,2
...,...,...,...,...,...,...,...,...,...,...
250,protection from enemies,TFIDF,3,0.00,Bible,103,By David Bless the Lord O my soul O Lord my Go...,False,D,0
322,protection from enemies,TFIDF_GLoVe,4,22.61,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",False,A,1
323,protection from enemies,BERT,4,99.57,Psalter,42,"Judge me, God, and give judgment in my cause a...",False,B,0
324,protection from enemies,SBERT,4,96.26,Bible,53,For the End in hymns concerning understanding ...,False,C,3


In [15]:
common_queries = list(
    set(human_scores['Query'].unique()) &
    set(human_seen_results['Query'].unique())
)
common_queries

['praise in times of suffering',
 'Have mercy on me, O God, have mercy on me. For my soul trusts in Thee, and in the shadow of Thy wings will I hope, until iniquity pass away.',
 'Verses where the psalmist remembers past deliverance and uses it to find hope in present trials.',
 'How does the psalmist express trust in God while surrounded by fear and uncertainty?',
 'For the Peace of the world',
 'mercy',
 'Create in me a clean heart',
 'protection from enemies',
 'prayer',
 'The Lord is my shepherd',
 'Rejoice, O ye heavens, sound the trumpets, ye foundation of the earth, thunder forth gladness, O ye mountains: for behold, Emmanuel to the Cross our sins, and the Giver of Life hath slain death, raising up Adam; for He loveth mankind.']

In [17]:
keys = ['Query', 'Method', 'Rank']

In [18]:
human_scores[
    human_scores.duplicated(subset=keys, keep=False)
].sort_values(keys)

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,User,Score
5,6,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,caden,4
230,259,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,p03,1
231,260,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,p06,8
232,261,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,p02,8
6,7,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.06,2,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",caden,8
...,...,...,...,...,...,...,...,...,...,...,...
631,696,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,22.61,4,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",p08,7
139,152,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,caden,6
632,697,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,p02,9
633,698,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,p01,5


In [19]:
# Make sure there is only one LLM score per result
llm_scores = human_seen_llm_results[
    keys + ['Score']
].drop_duplicates(
    subset=keys,
    keep='first'
)

# Add LLM score to the human scores
human_scores_combined = human_scores.merge(
    llm_scores,
    on=keys,
    how='left',
    validate='many_to_one'
)

# Rename the LLM score
human_scores_combined = human_scores_combined.rename(
    columns={'Score': 'LLM_Score'}
)

print("Human scores:", len(human_scores))
print("Combined:", len(human_scores_combined))

Human scores: 860
Combined: 860


In [20]:
human_scores.groupby(keys).size().value_counts().sort_index()

4    215
Name: count, dtype: int64

## Looking at overlap

In [22]:
human_scores

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,User,Score
0,0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...,...
855,941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p09,8
856,942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p03,1
857,943,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2
858,944,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7


In [23]:
import numpy as np 
common = human_scores.merge(
    llm_scores,
    on=['Query', 'Method', 'Rank'],
    how='inner'
)

common = common.rename(columns={'Score_y': "llm_score"})
# common

In [24]:
def score_10_to_3(score):
    if score <= 2:
        return 0
    elif score <= 5:
        return 1
    elif score <= 7:
        return 2
    else:
        return 3

In [25]:
common['score_x_condenced'] = common['Score_x'].apply(score_10_to_3)
common.columns

Index(['Unnamed: 0', 'Query', 'Query Category', 'Method',
       'Similarity Score (%)', 'Rank', 'Text', 'Psalm Num', 'Verse', 'User',
       'Score_x', 'llm_score', 'score_x_condenced'],
      dtype='str')

In [26]:
human_wide = (
    common
    .pivot_table(
        index=['Query', 'Query Category', 'Method', 'Similarity Score (%)', 'Rank', 'Text', 'Psalm Num', 'Verse'],
        columns='User',
        values='score_x_condenced',
        aggfunc='first'
    )
    .reset_index()
)

In [27]:
human_wide

User,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,caden,p01,...,p04,p05,p06,p07,p08,p09,p10,p13,p16,p17
0,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.31,5,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,1.0,NaN,...,NaN,2.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.66,4,Psalter,43,"We have heard with our ears, O God, for our fa...",0.0,1.0,...,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.69,3,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",1.0,NaN,...,NaN,3.0,2.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN
3,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.06,2,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",3.0,NaN,...,NaN,NaN,1.0,NaN,2.0,NaN,1.0,NaN,NaN,NaN
4,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,1.0,NaN,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,2.0,1.0,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,22.61,4,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",1.0,NaN,...,NaN,NaN,0.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN
190,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,23.85,3,Psalter,86,His foundations are in the holy mountains. The...,1.0,NaN,...,1.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN
191,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,25.01,2,Bible,28,A psalm by David the final day of the Feast of...,3.0,1.0,...,NaN,NaN,0.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN


In [38]:
scores = human_wide
scores

User,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,caden,p01,...,p04,p05,p06,p07,p08,p09,p10,p13,p16,p17
0,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.31,5,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,1.0,NaN,...,NaN,2.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.66,4,Psalter,43,"We have heard with our ears, O God, for our fa...",0.0,1.0,...,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.69,3,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",1.0,NaN,...,NaN,3.0,2.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN
3,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.06,2,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",3.0,NaN,...,NaN,NaN,1.0,NaN,2.0,NaN,1.0,NaN,NaN,NaN
4,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,1.0,NaN,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,2.0,1.0,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,22.61,4,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",1.0,NaN,...,NaN,NaN,0.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN
190,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,23.85,3,Psalter,86,His foundations are in the holy mountains. The...,1.0,NaN,...,1.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN
191,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,25.01,2,Bible,28,A psalm by David the final day of the Feast of...,3.0,1.0,...,NaN,NaN,0.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN


In [39]:
llm_scores

,Query,Method,Rank,Score
79,Create in me a clean heart,TFIDF_GLoVe,3,3
80,Create in me a clean heart,BERT,3,1
81,Create in me a clean heart,SBERT,3,2
82,Create in me a clean heart,TFIDF,3,0
102,Create in me a clean heart,TFIDF_GLoVe,2,2
...,...,...,...,...
250,protection from enemies,TFIDF,3,0
322,protection from enemies,TFIDF_GLoVe,4,1
323,protection from enemies,BERT,4,0
324,protection from enemies,SBERT,4,3


In [42]:
scores = scores.merge(llm_scores[['Query', 'Method', 'Rank', 'Score']],
                     on=['Query', 'Method', 'Rank'],
                     how='left'
                    )

In [45]:
scores = scores.rename(columns={'Score':'llm_score'})

In [46]:
scores.to_csv('human_and_llm_scores.csv')

* we can confirm that alll of the query caetorgiers are properly represented within the data based on the queries used and the corresponding query categories
* by working and adjusting the math for the values counts we can see that all the data is represented correctly

============================================